In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================
# PAPER 1
# DenseNet121 Molecular Subtype Classification
# Version 1
# ============================================================

print("=" * 80)
print("Paper 1 : DenseNet121 Breast Cancer Molecular Subtype Classification")
print("=" * 80)

# ============================================================
# STANDARD LIBRARIES
# ============================================================

import os
import gc
import time
import copy
import random
import warnings
from pathlib import Path

# ============================================================
# SCIENTIFIC LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

# ============================================================
# PYTORCH
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

from torchvision import datasets
from torchvision import transforms
from torchvision.models import densenet121
from torchvision.models import DenseNet121_Weights

# ============================================================
# SCIKIT-LEARN
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    balanced_accuracy_score
)

# ============================================================
# MIXED PRECISION
# ============================================================

from torch.cuda.amp import (
    autocast,
    GradScaler
)

# ============================================================
# WARNINGS
# ============================================================

warnings.filterwarnings("ignore")

print("\nLibraries imported successfully.")
# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

IMAGE_SIZE = 224

NUM_CLASSES = 4

BATCH_SIZE = 64

NUM_WORKERS = min(4, os.cpu_count())

EPOCHS = 35

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.1

EARLY_STOPPING_PATIENCE = 8

GRADIENT_CLIP = 1.0

USE_AMP = True

PIN_MEMORY = True

PERSISTENT_WORKERS = True

MODEL_NAME = "DenseNet121"



# ============================================================
# ROOT DIRECTORIES
# ============================================================

ROOT = Path("/kaggle/input/datasets")

ANISHA = ROOT / "anishapanja"

COAUTHOR = ROOT / "hritishachoudhury"

MANIFEST_ROOT = ANISHA / "bc-xai-final-manifest"
WORK_DIR = Path("/kaggle/working")

WORK_DIR.mkdir(exist_ok=True)

# ============================================================
# DATASET PATHS
# ============================================================

A2_PATH = ANISHA / "bc-xai-a2-2000-partial"

E2_PATH = ANISHA / "bc-xai-e2-2000"

C8_BATCHES = [

    ANISHA / "bc-xai-c8-patches-batch1",

    ANISHA / "bc-xai-c8-patches-batch2",

    ANISHA / "bc-xai-c8-patches-batch3",

    ANISHA / "bc-xai-c8-patches-batch4"

]

A8_BATCHES = [

    ANISHA / "bc-xai-a8-patches-batch1",

    ANISHA / "bc-xai-a8-patches-batch2",

    ANISHA / "bc-xai-a8-patches-batch3",

    ANISHA / "bc-xai-a8-patches-batch4",

    ANISHA / "bc-xai-a8-patches-batch5"

]

D8_PATH = COAUTHOR / "bc-xai-d8-patches"

BH_PATH = COAUTHOR / "bc-xai-bh-patches"

# ============================================================
# VERIFY DATASETS
# ============================================================

ALL_DATASETS = [

    ("A2", A2_PATH),

    ("E2", E2_PATH),

    ("D8", D8_PATH),

    ("BH", BH_PATH)

]

for ds in C8_BATCHES:

    ALL_DATASETS.append((ds.name, ds))

for ds in A8_BATCHES:

    ALL_DATASETS.append((ds.name, ds))

print("=" * 80)
print("VERIFYING ATTACHED KAGGLE DATASETS")
print("=" * 80)

missing = []

for name, path in ALL_DATASETS:

    exists = path.exists()

    print(f"{name:<35} {exists}")

    if not exists:

        missing.append(name)

if len(missing):

    raise FileNotFoundError(
        f"\nMissing Kaggle datasets:\n{missing}"
    )

print("\nAll datasets found successfully.")

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print("\nDevice :", DEVICE)

if DEVICE.type == "cuda":

    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 80)
# ============================================================
# REPRODUCIBILITY
# ============================================================

def seed_everything(seed: int = 42):

    random.seed(seed)

    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False

    torch.use_deterministic_algorithms(False)


seed_everything(SEED)

print("=" * 80)
print("Random Seed Fixed")
print("=" * 80)

print(f"Seed : {SEED}")

# ============================================================
# GPU INFORMATION
# ============================================================

print("\n")
print("=" * 80)
print("SYSTEM INFORMATION")
print("=" * 80)

print(f"PyTorch Version : {torch.__version__}")

print(f"CUDA Available : {torch.cuda.is_available()}")

print(f"Device : {DEVICE}")

if torch.cuda.is_available():

    print(f"GPU : {torch.cuda.get_device_name(0)}")

    print(
        f"CUDA Version : {torch.version.cuda}"
    )

    gpu_properties = torch.cuda.get_device_properties(0)

    print(
        f"Total GPU Memory : {gpu_properties.total_memory / 1024**3:.2f} GB"
    )

print("=" * 80)
# ============================================================
# LOAD FINAL MANIFEST
# ============================================================

print("=" * 80)
print("LOADING FINAL MANIFEST")
print("=" * 80)

MANIFEST_ROOT = ANISHA / "bc-xai-final-manifest"

PATCH_CSV = MANIFEST_ROOT / "patches_with_split_final.csv"

PATIENT_CSV = MANIFEST_ROOT / "patient_summary.csv"

if not PATCH_CSV.exists():
    raise FileNotFoundError(f"Cannot find:\n{PATCH_CSV}")

if not PATIENT_CSV.exists():
    raise FileNotFoundError(f"Cannot find:\n{PATIENT_CSV}")

patch_df = pd.read_csv(PATCH_CSV)

patch_df["subtype_clean"] = (
    patch_df["subtype_clean"]
    .astype(str)
    .str.strip()
    .replace({
        "Her2": "HER2"
    })
)


patch_df["dataset"] = (
    patch_df["dataset"]
    .astype(str)
    .str.strip()
)

patient_df = pd.read_csv(PATIENT_CSV)

print(f"Patch CSV Loaded    : {PATCH_CSV.name}")
print(f"Patient CSV Loaded  : {PATIENT_CSV.name}")

print()

print("Patch DataFrame Shape :", patch_df.shape)

print("Patient DataFrame Shape :", patient_df.shape)

# ============================================================
# VERIFY REQUIRED COLUMNS
# ============================================================

required_columns = [

    "patient_id",

    "site",

    "subtype",

    "subtype_clean",

    "dataset",

    "relative_path",

    "patch_name",

    "split"

]

missing_columns = [

    col

    for col in required_columns

    if col not in patch_df.columns

]

if len(missing_columns):

    raise ValueError(

        f"Missing Columns:\n{missing_columns}"

    )

print("\nAll required columns present.")

# ============================================================
# LABEL ENCODING
# ============================================================

patch_df["subtype_clean"] = (
    patch_df["subtype_clean"]
    .astype(str)
    .str.strip()
)

CLASS_TO_INDEX = {
    "Basal": 0,
    "HER2": 1,
    "LumA": 2,
    "LumB": 3
}

INDEX_TO_CLASS = {
    v: k
    for k, v in CLASS_TO_INDEX.items()
}

print("\nUnique subtype labels in manifest:")
print(sorted(patch_df["subtype_clean"].unique()))

patch_df["label"] = patch_df["subtype_clean"].map(CLASS_TO_INDEX)

unknown_mask = patch_df["label"].isna()

if unknown_mask.any():

    print("\nUnknown subtype labels found:")
    print(
        patch_df.loc[
            unknown_mask,
            "subtype_clean"
        ].value_counts(dropna=False)
    )

    raise ValueError("Unknown subtype labels detected.")

patch_df["label"] = patch_df["label"].astype(int)
# ============================================================
# CLASS NAMES (ORDER MATCHES LABEL INDICES)
# ============================================================

CLASS_NAMES = [

    INDEX_TO_CLASS[i]

    for i in range(len(INDEX_TO_CLASS))

]

print("Class Mapping:")

for idx, class_name in enumerate(CLASS_NAMES):

    print(f"{idx}: {class_name}")
# ============================================================
# TRAIN / VALIDATION / TEST
# ============================================================

train_df = (

    patch_df

    [patch_df["split"]=="train"]

    .reset_index(drop=True)

)

val_df = (

    patch_df

    [patch_df["split"]=="val"]

    .reset_index(drop=True)

)

test_df = (

    patch_df

    [patch_df["split"]=="test"]

    .reset_index(drop=True)

)

# ============================================================
# SUMMARY
# ============================================================

print("\n")

print("=" * 80)

print("DATASET SUMMARY")

print("=" * 80)

print(f"Training Patches   : {len(train_df):,}")

print(f"Validation Patches : {len(val_df):,}")

print(f"Testing Patches    : {len(test_df):,}")

print()

print("Training Class Distribution")

print(

    train_df["subtype_clean"]

    .value_counts()

    .sort_index()

)

print()

print("Validation Class Distribution")

print(

    val_df["subtype_clean"]

    .value_counts()

    .sort_index()

)

print()

print("Testing Class Distribution")

print(

    test_df["subtype_clean"]

    .value_counts()

    .sort_index()

)

print()

print("=" * 80)

print("Manifest Loaded Successfully")

print("=" * 80)

# ============================================================
# IMAGE TRANSFORMS
# ============================================================

train_transform = transforms.Compose([

    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.80, 1.00),
        ratio=(0.90, 1.10)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomRotation(degrees=15),

    transforms.ColorJitter(
        brightness=0.20,
        contrast=0.20,
        saturation=0.15,
        hue=0.02
    ),

    transforms.RandomAutocontrast(p=0.20),

    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1, 1.5)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])


val_transform = transforms.Compose([

    transforms.Resize(256),

    transforms.CenterCrop(IMAGE_SIZE),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])

test_transform = val_transform


# ============================================================
# DATASET ROOT MAPPING
# ============================================================

DATASET_ROOTS = {

    "bc-xai-a2-2000-partial": A2_PATH,

    "bc-xai-e2-2000": E2_PATH,

    "bc-xai-d8-patches": D8_PATH,

    "bc-xai-bh-patches": BH_PATH,

    "bc-xai-c8-patches-batch1": C8_BATCHES[0],
    "bc-xai-c8-patches-batch2": C8_BATCHES[1],
    "bc-xai-c8-patches-batch3": C8_BATCHES[2],
    "bc-xai-c8-patches-batch4": C8_BATCHES[3],

    "bc-xai-a8-patches-batch1": A8_BATCHES[0],
    "bc-xai-a8-patches-batch2": A8_BATCHES[1],
    "bc-xai-a8-patches-batch3": A8_BATCHES[2],
    "bc-xai-a8-patches-batch4": A8_BATCHES[3],
    "bc-xai-a8-patches-batch5": A8_BATCHES[4]

}


# ============================================================
# CUSTOM DATASET
# ============================================================

class BreastCancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)

        self.transform = transform


    def __len__(self):

        return len(self.df)


    def __getitem__(self, index):

        row = self.df.iloc[index]

        dataset_name = str(row["dataset"]).strip()

        relative_path = Path(str(row["relative_path"]).strip())

        if dataset_name not in DATASET_ROOTS:

            raise ValueError(
                f"Unknown dataset '{dataset_name}'."
            )

        image_path = DATASET_ROOTS[dataset_name] / relative_path

        if not image_path.exists():

            raise FileNotFoundError(

                f"""
Image not found.

Dataset      : {dataset_name}

Relative Path: {relative_path}

Full Path    : {image_path}
"""
            )

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:

            image = self.transform(image)

        label = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        sample = {

            "image": image,

            "label": label,

            "patient_id": row["patient_id"],

            "site": row["site"],

            "subtype": row["subtype_clean"],

            "dataset": dataset_name,

            "path": str(image_path)

        }

        return sample


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = BreastCancerDataset(

    dataframe=train_df,

    transform=train_transform

)

val_dataset = BreastCancerDataset(

    dataframe=val_df,

    transform=val_transform

)

test_dataset = BreastCancerDataset(

    dataframe=test_df,

    transform=test_transform

)


# ============================================================
# DATASET SUMMARY
# ============================================================

print("=" * 80)

print("DATASETS CREATED")

print("=" * 80)

print(f"Training Samples   : {len(train_dataset):,}")

print(f"Validation Samples : {len(val_dataset):,}")

print(f"Testing Samples    : {len(test_dataset):,}")

print("=" * 80)
# ============================================================
# COMPUTE CLASS WEIGHTS
# ============================================================

print("=" * 80)
print("COMPUTING CLASS WEIGHTS")
print("=" * 80)

train_labels = train_df["label"].values

class_counts = np.bincount(
    train_labels,
    minlength=NUM_CLASSES
)

class_weights = 1.0 / class_counts

sample_weights = class_weights[train_labels]

sample_weights = torch.DoubleTensor(sample_weights)

weighted_sampler = WeightedRandomSampler(

    weights=sample_weights,

    num_samples=len(sample_weights),

    replacement=True

)

print("Training Class Counts")

for idx, count in enumerate(class_counts):

    print(f"{INDEX_TO_CLASS[idx]:<6} : {count:,}")

print()

print("Class Weights")

for idx, weight in enumerate(class_weights):

    print(f"{INDEX_TO_CLASS[idx]:<6} : {weight:.6f}")

print("=" * 80)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(

    dataset=train_dataset,

    batch_size=BATCH_SIZE,

    sampler=weighted_sampler,

    num_workers=NUM_WORKERS,

    pin_memory=True,

    persistent_workers=NUM_WORKERS > 0,

    drop_last=True

)


val_loader = DataLoader(

    dataset=val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=True,

    persistent_workers=NUM_WORKERS > 0

)


test_loader = DataLoader(

    dataset=test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=True,

    persistent_workers=NUM_WORKERS > 0

)


# ============================================================
# VERIFY DATALOADERS
# ============================================================

print("\n")

print("=" * 80)
print("DATALOADER SUMMARY")
print("=" * 80)

print(f"Training Batches   : {len(train_loader):,}")

print(f"Validation Batches : {len(val_loader):,}")

print(f"Testing Batches    : {len(test_loader):,}")

print("=" * 80)
# ============================================================
# BUILD DENSENET121 MODEL
# ============================================================

print("=" * 80)
print("BUILDING DENSENET121")
print("=" * 80)

model = densenet121(

    weights=DenseNet121_Weights.IMAGENET1K_V1

)

num_features = model.classifier.in_features

model.classifier = nn.Linear(

    in_features=num_features,

    out_features=NUM_CLASSES

)

model = model.to(DEVICE)

print(model)

print()

print(f"Feature Dimension : {num_features}")

print(f"Number of Classes : {NUM_CLASSES}")

total_params = sum(

    p.numel()

    for p in model.parameters()

)

trainable_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)

print(f"Total Parameters      : {total_params:,}")

print(f"Trainable Parameters  : {trainable_params:,}")

print("=" * 80)


# ============================================================
# VERIFY MODEL
# ============================================================

dummy_input = torch.randn(

    2,

    3,

    IMAGE_SIZE,

    IMAGE_SIZE,

    device=DEVICE

)

with torch.no_grad():

    dummy_output = model(dummy_input)

print()

print("Model Verification")

print(f"Input Shape  : {dummy_input.shape}")

print(f"Output Shape : {dummy_output.shape}")

assert dummy_output.shape == (2, NUM_CLASSES)

print("Forward Pass Successful")

print("=" * 80)
# ============================================================
# LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss(

    label_smoothing=LABEL_SMOOTHING

)

# ============================================================
# OPTIMIZER
# ============================================================

decay = []

no_decay = []

for name, param in model.named_parameters():

    if not param.requires_grad:
        continue

    if param.ndim == 1 or name.endswith(".bias"):
        no_decay.append(param)

    else:
        decay.append(param)

optimizer = optim.AdamW(

    [

        {

            "params": decay,

            "weight_decay": WEIGHT_DECAY

        },

        {

            "params": no_decay,

            "weight_decay": 0.0

        }

    ],

    lr=LEARNING_RATE

)

# ============================================================
# LEARNING RATE SCHEDULER
# ============================================================

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(

    optimizer=optimizer,

    T_0=10,

    T_mult=2,

    eta_min=1e-6

)

# ============================================================
# AUTOMATIC MIXED PRECISION (AMP)
# ============================================================

scaler = GradScaler(enabled=torch.cuda.is_available())

# ============================================================
# TRAINING STATE
# ============================================================

best_val_accuracy = 0.0

best_val_f1 = 0.0

best_epoch = -1

epochs_without_improvement = 0

history = {

    "train_loss": [],

    "train_accuracy": [],

    "val_loss": [],

    "val_accuracy": [],

    "val_f1": [],

    "learning_rate": []

}

# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print("=" * 80)
print("TRAINING CONFIGURATION")
print("=" * 80)

print(f"Loss Function        : CrossEntropyLoss")
print(f"Label Smoothing      : {LABEL_SMOOTHING}")

print()

print(f"Optimizer            : AdamW")
print(f"Learning Rate        : {LEARNING_RATE}")
print(f"Weight Decay         : {WEIGHT_DECAY}")

print()

print("Scheduler            : CosineAnnealingWarmRestarts")
print("T_0                  : 10")
print("T_mult               : 2")
print("Minimum LR           : 1e-6")

print()

print(f"Mixed Precision      : {torch.cuda.is_available()}")

print()

print(f"Early Stopping       : {EARLY_STOPPING_PATIENCE} epochs")

print("=" * 80)
# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    criterion,
    optimizer,
    scheduler,
    scaler,
    device,
    epoch
):

    model.train()

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    progress_bar = tqdm(

        dataloader,

        desc=f"Epoch {epoch + 1}/{EPOCHS} [Train]",

        leave=False

    )

    for batch in progress_bar:

        images = batch["image"].to(device, non_blocking=True)

        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=torch.cuda.is_available()):

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            GRADIENT_CLIP

        )

        scaler.step(optimizer)

        scaler.update()

        predictions = outputs.argmax(dim=1)

        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size

        running_correct += (predictions == labels).sum().item()

        total_samples += batch_size

        current_loss = running_loss / total_samples

        current_accuracy = 100.0 * running_correct / total_samples

        current_lr = optimizer.param_groups[0]["lr"]

        progress_bar.set_postfix({

            "Loss": f"{current_loss:.4f}",

            "Acc": f"{current_accuracy:.2f}%",

            "LR": f"{current_lr:.2e}"

        })

    scheduler.step(epoch + 1)

    epoch_loss = running_loss / total_samples

    epoch_accuracy = running_correct / total_samples

    return {

    "loss": epoch_loss,

    "accuracy": epoch_accuracy,

    "learning_rate": optimizer.param_groups[0]["lr"]

}
# ============================================================
# VALIDATION FUNCTION
# ============================================================

def validate_one_epoch(
    model,
    dataloader,
    criterion,
    device,
):

    model.eval()

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    all_labels = []
    all_predictions = []
    all_probabilities = []

    progress_bar = tqdm(
        dataloader,
        desc="Validation",
        leave=False,
    )

    with torch.no_grad():

        for batch in progress_bar:

            images = batch["image"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)

            with autocast(enabled=torch.cuda.is_available()):

                outputs = model(images)

                loss = criterion(outputs, labels)

            probabilities = torch.softmax(outputs, dim=1)

            predictions = torch.argmax(outputs, dim=1)

            batch_size = labels.size(0)

            running_loss += loss.item() * batch_size

            running_correct += (predictions == labels).sum().item()

            total_samples += batch_size

            all_labels.extend(labels.cpu().numpy())

            all_predictions.extend(predictions.cpu().numpy())

            all_probabilities.extend(probabilities.cpu().numpy())

            avg_loss = running_loss / total_samples

            avg_acc = 100.0 * running_correct / total_samples

            progress_bar.set_postfix({

                "Loss": f"{avg_loss:.4f}",

                "Acc": f"{avg_acc:.2f}%"

            })

    epoch_loss = running_loss / total_samples

    epoch_accuracy = running_correct / total_samples

    macro_f1 = f1_score(

        all_labels,

        all_predictions,

        average="macro"

    )
    macro_precision = precision_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
    )

    macro_recall = recall_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
    )

    balanced_accuracy = balanced_accuracy_score(
    all_labels,
    all_predictions
    )
    return {

        "loss": epoch_loss,

        "accuracy": epoch_accuracy,

        "macro_f1": macro_f1,

        "labels": np.array(all_labels),

        "predictions": np.array(all_predictions),

        "probabilities": np.array(all_probabilities),
        "balanced_accuracy": balanced_accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
    }
    
# ============================================================
# MODEL TRAINING
# ============================================================

print("=" * 80)
print("STARTING MODEL TRAINING")
print("=" * 80)

BEST_MODEL_PATH = WORK_DIR / "best_densenet121.pth"
LAST_MODEL_PATH = WORK_DIR / "last_densenet121.pth"
# Resume checkpoint location
RESUME_LAST_MODEL = Path(
    "/kaggle/input/datasets/anishapanja/densenet121-v1/last_densenet121.pth"
)
start_epoch = 0

# --------------------------------------------------------
# Resume Training (if checkpoint exists)
# --------------------------------------------------------

if RESUME_LAST_MODEL.exists():

    print("=" * 80)
    print("RESUMING TRAINING FROM LAST CHECKPOINT")
    print("=" * 80)

    checkpoint = torch.load(
        RESUME_LAST_MODEL,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    start_epoch = checkpoint["epoch"]

    best_val_accuracy = checkpoint["best_val_accuracy"]

    best_val_f1 = checkpoint["best_val_f1"]

    best_epoch = checkpoint["best_epoch"]

    epochs_without_improvement = checkpoint["epochs_without_improvement"]

    history = checkpoint["history"]

    print(f"Resuming from Epoch {start_epoch + 1}")

else:

    print("Starting training from scratch.")

for epoch in range(start_epoch, EPOCHS):

    epoch_start = time.time()

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    train_results = train_one_epoch(

        model=model,

        dataloader=train_loader,

        criterion=criterion,

        optimizer=optimizer,

        scheduler=scheduler,

        scaler=scaler,

        device=DEVICE,

        epoch=epoch

    )

    # --------------------------------------------------------
    # Validate
    # --------------------------------------------------------

    val_results = validate_one_epoch(

        model=model,

        dataloader=val_loader,

        criterion=criterion,

        device=DEVICE

    )

    epoch_time = time.time() - epoch_start

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_results['loss']:.4f} | Train Acc: {train_results['accuracy']*100:.2f}% | Val Loss: {val_results['loss']:.4f} | Val Acc: {val_results['accuracy']*100:.2f}% | Val F1: {val_results['macro_f1']:.4f} | Time: {epoch_time/60:.1f} min")


    # --------------------------------------------------------
    # Update History
    # --------------------------------------------------------

    history["train_loss"].append(train_results["loss"])

    history["train_accuracy"].append(train_results["accuracy"])

    history["val_loss"].append(val_results["loss"])

    history["val_accuracy"].append(val_results["accuracy"])

    history["val_f1"].append(val_results["macro_f1"])

    history["learning_rate"].append(

        train_results["learning_rate"]

    )

    # --------------------------------------------------------
    # Checkpoint Logic
    # --------------------------------------------------------

    improved = (

        val_results["macro_f1"] > best_val_f1

        or

        (

            np.isclose(

                val_results["macro_f1"],

                best_val_f1

            )

            and

            val_results["accuracy"] > best_val_accuracy

        )

    )

    if improved:

        best_val_f1 = val_results["macro_f1"]

        best_val_accuracy = val_results["accuracy"]

        best_epoch = epoch + 1

        epochs_without_improvement = 0

        torch.save(

            {

                "epoch": epoch+1,

                "model_state_dict": model.state_dict(),

                "optimizer_state_dict": optimizer.state_dict(),

                "scheduler_state_dict": scheduler.state_dict(),

                "best_val_accuracy": best_val_accuracy,

                "best_val_f1": best_val_f1,

                "best_epoch": best_epoch,

                "epochs_without_improvement": epochs_without_improvement,

                "history": history

            },

            BEST_MODEL_PATH

        )
        status = "Improved"

    else:

        epochs_without_improvement += 1
# --------------------------------------------------------
# SAVE LATEST CHECKPOINT
# --------------------------------------------------------

    torch.save(

        {

            "epoch": epoch + 1,

            "model_state_dict": model.state_dict(),

            "optimizer_state_dict": optimizer.state_dict(),

            "scheduler_state_dict": scheduler.state_dict(),

            "best_val_accuracy": best_val_accuracy,

            "best_val_f1": best_val_f1,

            "best_epoch": best_epoch,

            "epochs_without_improvement": epochs_without_improvement,

            "history": history

        },

        LAST_MODEL_PATH

    )

 #=========================================
# TRAINING HISTORY VISUALIZATION
# ============================================================

epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(

    2,
    2,

    figsize=(16, 12)

)

# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

axes[0, 0].plot(

    epochs,

    history["train_loss"],

    marker="o",

    linewidth=2,

    label="Train"

)

axes[0, 0].plot(

    epochs,

    history["val_loss"],

    marker="s",

    linewidth=2,

    label="Validation"

)

axes[0, 0].set_title("Loss")

axes[0, 0].set_xlabel("Epoch")

axes[0, 0].set_ylabel("Cross-Entropy Loss")

axes[0, 0].grid(True)

axes[0, 0].legend()

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

axes[0, 1].plot(

    epochs,

    np.array(history["train_accuracy"]) * 100,

    marker="o",

    linewidth=2,

    label="Train"

)

axes[0, 1].plot(

    epochs,

    np.array(history["val_accuracy"]) * 100,

    marker="s",

    linewidth=2,

    label="Validation"

)

axes[0, 1].set_title("Accuracy")

axes[0, 1].set_xlabel("Epoch")

axes[0, 1].set_ylabel("Accuracy (%)")

axes[0, 1].grid(True)

axes[0, 1].legend()

# ------------------------------------------------------------
# Macro F1
# ------------------------------------------------------------

axes[1, 0].plot(

    epochs,

    history["val_f1"],

    marker="o",

    linewidth=2,

    color="green"

)

axes[1, 0].set_title("Validation Macro F1")

axes[1, 0].set_xlabel("Epoch")

axes[1, 0].set_ylabel("Macro F1")

axes[1, 0].grid(True)

# ------------------------------------------------------------
# Learning Rate
# ------------------------------------------------------------

axes[1, 1].plot(

    epochs,

    history["learning_rate"],

    marker="o",

    linewidth=2,

    color="red"

)

axes[1, 1].set_title("Learning Rate Schedule")

axes[1, 1].set_xlabel("Epoch")

axes[1, 1].set_ylabel("Learning Rate")

axes[1, 1].set_yscale("log")

axes[1, 1].grid(True)

plt.tight_layout()

save_path = WORK_DIR / "densenet121_training_history.png"

plt.savefig(

    save_path,

    dpi=300,

    bbox_inches="tight"

)

plt.show()

print()

print("=" * 80)

print("Training history figure saved.")

print(save_path)

print("=" * 80)
# ============================================================
# HELD-OUT TEST SET EVALUATION
# ============================================================

print("=" * 80)
print("EVALUATING BEST MODEL ON HELD-OUT TEST SET")
print("=" * 80)

test_results = validate_one_epoch(

    model=model,

    dataloader=test_loader,

    criterion=criterion,

    device=DEVICE

)

print()

print("=" * 80)
print("TEST SET PERFORMANCE")
print("=" * 80)

print(f"Test Loss      : {test_results['loss']:.4f}")

print(f"Test Accuracy  : {test_results['accuracy']*100:.2f}%")

print(f"Test Macro F1  : {test_results['macro_f1']:.4f}")

print("=" * 80)

# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

test_labels = test_results["labels"]

test_predictions = test_results["predictions"]

test_probabilities = test_results["probabilities"]
# Maximum predicted probability for each test sample
test_confidence = np.max(
    test_probabilities,
    axis=1
)
    # ============================================================
# CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import classification_report

print("=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

report = classification_report(

    test_labels,

    test_predictions,

    target_names=CLASS_NAMES,

    digits=4,

    zero_division=0,

    output_dict=True

)

report_df = pd.DataFrame(report).transpose()

display(report_df.round(4))

report_path = WORK_DIR / "classification_report.csv"

report_df.to_csv(

    report_path,

    index=True

)

print()

print("=" * 80)

print(f"Classification report saved to:\n{report_path}")

print("=" * 80)
# ============================================================
# CONFUSION MATRIX
# ============================================================

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(

    test_labels,

    test_predictions

)

# ------------------------------------------------------------
# Plot Confusion Matrix
# ------------------------------------------------------------

plt.figure(figsize=(8, 7))

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    cmap="Blues",

    xticklabels=CLASS_NAMES,

    yticklabels=CLASS_NAMES,

    linewidths=0.5,

    cbar=True

)

plt.title(
    "DenseNet121 Confusion Matrix",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel(
    "Predicted Class",
    fontsize=13
)

plt.ylabel(
    "True Class",
    fontsize=13
)

plt.xticks(rotation=20)

plt.yticks(rotation=0)

plt.tight_layout()

figure_path = WORK_DIR / "densenet121_confusion_matrix.png"

plt.savefig(

    figure_path,

    dpi=300,

    bbox_inches="tight"

)

plt.show()

print()

print("=" * 80)

print(f"Confusion matrix saved to:\n{figure_path}")

print("=" * 80)
# ============================================================
# MULTI-CLASS ROC CURVE
# ============================================================

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# ------------------------------------------------------------
# One-vs-Rest Label Binarization
# ------------------------------------------------------------

y_true = label_binarize(

    test_labels,

    classes=np.arange(NUM_CLASSES)

)

# ------------------------------------------------------------
# Compute ROC Curve for Each Class
# ------------------------------------------------------------

fpr = {}

tpr = {}

roc_auc = {}

for i in range(NUM_CLASSES):

    fpr[i], tpr[i], _ = roc_curve(

        y_true[:, i],

        test_probabilities[:, i]

    )

    roc_auc[i] = auc(

        fpr[i],

        tpr[i]

    )

# ------------------------------------------------------------
# Plot ROC Curves
# ------------------------------------------------------------

plt.figure(figsize=(8, 7))

for i in range(NUM_CLASSES):

    plt.plot(

        fpr[i],

        tpr[i],

        linewidth=2,

        label=f"{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.3f})"

    )

plt.plot(

    [0, 1],

    [0, 1],

    linestyle="--",

    color="gray"

)

plt.xlabel(

    "False Positive Rate",

    fontsize=13

)

plt.ylabel(

    "True Positive Rate",

    fontsize=13

)

plt.title(

    "DenseNet121 ROC Curves",

    fontsize=16,

    fontweight="bold"

)

plt.legend(

    loc="lower right"

)

plt.grid(alpha=0.3)

plt.tight_layout()

figure_path = WORK_DIR / "densenet121_roc_curve.png"

plt.savefig(

    figure_path,

    dpi=300,

    bbox_inches="tight"

)

plt.show()

print()

print("=" * 80)

print("Per-Class ROC-AUC")

print("=" * 80)

for i in range(NUM_CLASSES):

    print(

        f"{CLASS_NAMES[i]:<15}: {roc_auc[i]:.4f}"

    )

print("=" * 80)
# ============================================================
# MULTI-CLASS PRECISION-RECALL CURVES
# ============================================================

from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_curve, average_precision_score

# ------------------------------------------------------------
# Binarize Labels
# ------------------------------------------------------------

y_true = label_binarize(
    test_labels,
    classes=np.arange(NUM_CLASSES)
)

# ------------------------------------------------------------
# Compute Precision-Recall Curves
# ------------------------------------------------------------

precision = {}
recall = {}
average_precision = {}

for i in range(NUM_CLASSES):

    precision[i], recall[i], _ = precision_recall_curve(

        y_true[:, i],

        test_probabilities[:, i]

    )

    average_precision[i] = average_precision_score(

        y_true[:, i],

        test_probabilities[:, i]

    )

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(8,7))

for i in range(NUM_CLASSES):

    plt.plot(

        recall[i],

        precision[i],

        linewidth=2,

        label=f"{CLASS_NAMES[i]} (AP = {average_precision[i]:.3f})"

    )

plt.xlabel("Recall", fontsize=13)

plt.ylabel("Precision", fontsize=13)

plt.title(

    "DenseNet121 Precision-Recall Curves",

    fontsize=16,

    fontweight="bold"

)

plt.grid(alpha=0.3)

plt.legend()

plt.tight_layout()

figure_path = WORK_DIR / "densenet121_precision_recall_curve.png"

plt.savefig(

    figure_path,

    dpi=300,

    bbox_inches="tight"

)

plt.show()

print()

print("="*80)

print("Average Precision Scores")

print("="*80)

for i in range(NUM_CLASSES):

    print(f"{CLASS_NAMES[i]:<12}: {average_precision[i]:.4f}")

print("="*80)
# ============================================================
# EXPORT TEST PREDICTIONS
# ============================================================

prediction_df = test_df.copy().reset_index(drop=True)

prediction_df["True_Label"] = test_labels

prediction_df["Predicted_Label"] = test_predictions

prediction_df["True_Class"] = [

    INDEX_TO_CLASS[i]

    for i in test_labels

]

prediction_df["Predicted_Class"] = [

    INDEX_TO_CLASS[i]

    for i in test_predictions

]

prediction_df["Confidence"] = test_confidence

for i, class_name in enumerate(CLASS_NAMES):

    prediction_df[f"Prob_{class_name}"] = test_probabilities[:, i]

prediction_file = WORK_DIR / "densenet121_predictions.csv"

prediction_df.to_csv(

    prediction_file,

    index=False

)

print(f"Prediction file saved to:\n{prediction_file}")
# ============================================================
# MISCLASSIFICATION ANALYSIS
# ============================================================

prediction_df["Correct"] = (

    prediction_df["True_Label"]

    ==

    prediction_df["Predicted_Label"]

)

correct_df = prediction_df[

    prediction_df["Correct"]

].copy()

incorrect_df = prediction_df[

    ~prediction_df["Correct"]

].copy()

print("="*80)

print(f"Total Samples      : {len(prediction_df)}")

print(f"Correct Predictions: {len(correct_df)}")

print(f"Misclassified      : {len(incorrect_df)}")

print("="*80)

print()

print("Top-20 Highest Confidence Errors")

display(

    incorrect_df

    .sort_values(

        "Confidence",

        ascending=False

    )

    .head(20)

)

print()

print("Top-20 Lowest Confidence Correct Predictions")

display(

    correct_df

    .sort_values(

        "Confidence",

        ascending=True

    )

    .head(20)

)
# ============================================================
# RUN SUMMARY
# ============================================================

summary = {

    "Architecture": "DenseNet121",

    "Best Epoch": best_epoch,

    "Validation Accuracy": best_val_accuracy,

    "Validation Macro F1": best_val_f1,

    "Test Accuracy": test_results["accuracy"],

    "Balanced Accuracy": test_results["balanced_accuracy"],

    "Macro Precision": test_results["macro_precision"],

    "Macro Recall": test_results["macro_recall"],

    "Macro F1": test_results["macro_f1"]

}

summary_df = pd.DataFrame(

    [summary]

)

display(summary_df)

summary_file = WORK_DIR / "densenet121_summary.csv"

summary_df.to_csv(

    summary_file,

    index=False

)

print()

print("="*80)

print("DenseNet121 experiment completed successfully.")

print(f"Summary saved to:\n{summary_file}")

print("="*80)
